RAG Pipeline - Data Ingestion to Vector DB

In [5]:
import os

from chromadb.utils import results
from langchain_classic import text_splitter
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from pathlib import Path

from langchain_core.tracers import langchain
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
#Read all the pdfs inside the directory

def process_all_pdfs(pdf_directory):
    "Process all the pdf files in a directory"
    all_documents = []
    pdf_dir = Path(pdf_directory)


    pdf_files = list(pdf_dir.glob('**/*.pdf'))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source-_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")


        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process
Processing ../data/pdf/ml_concepts.pdf
 Loaded 1 pages
Processing ../data/pdf/sample_resume.pdf
 Loaded 1 pages
Processing ../data/pdf/ai_basics.pdf
 Loaded 1 pages
Processing ../data/pdf/story.pdf
 Loaded 1 pages

Total documents loaded: 4


In [7]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-20T06:28:30+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-20T06:28:30+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/ml_concepts.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source-_file': 'ml_concepts.pdf', 'file_type': 'pdf'}, page_content='Machine Learning Concepts\nSupervised Learning: Linear Regression, Logistic Regression\nUnsupervised Learning: K-Means, PCA\nEvaluation Metrics: Accuracy, Precision, Recall'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-20T06:28:30+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-20T06:28:30+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/sample_resume.pdf', 'total_pages': 1, 'page'

In [8]:
def split_documents(documents,chunk_size=500,chunk_overlap=50):
    "Split a list of documents into chunks of size chunk_size"
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ',' ']
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:20]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [9]:
chunks = split_documents(all_pdf_documents)
chunks

Split 4 documents into 4 chunks

Example chunk:
Content: Machine Learning Con...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-20T06:28:30+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-20T06:28:30+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/ml_concepts.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source-_file': 'ml_concepts.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-20T06:28:30+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-20T06:28:30+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/ml_concepts.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source-_file': 'ml_concepts.pdf', 'file_type': 'pdf'}, page_content='Machine Learning Concepts\nSupervised Learning: Linear Regression, Logistic Regression\nUnsupervised Learning: K-Means, PCA\nEvaluation Metrics: Accuracy, Precision, Recall'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-20T06:28:30+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-20T06:28:30+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/sample_resume.pdf', 'total_pages': 1, 'page'

Embedding and vector store db


In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Tuple, Any

In [11]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager


embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9638.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


/var/folders/bm/rgcvk1q90f90sgsqss9sg60m0000gn/T/ipykernel_70580/1098155618.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [12]:
class VectorStore:
    """Manages document embeddings in ChromaDB VectorStore"""

    def __init__(self,collection_name:str= "pdf_documents",persist_directory: str= "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the chromadb client and collection"""

        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(self.persist_directory)

             # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description":"PDF document embedding for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing collection: {e}")
            raise

    def add_documents(self, documents:List[Any], embeddings:np.ndarray):
        """
        Add documents to the collection

        Args:
            documents: List of documents to add
            embeddings: numpy array of corresponding embeddings
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate uuid
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())


        try:
            #Add to collection
            self.collection.add(
                ids = ids,
                metadatas = metadatas,
                embeddings = embeddings_list,
                documents = documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vector_store = VectorStore()
vector_store



Vector store initialized. Collection: pdf_documents
Existing documents in collection: 4


In [13]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-20T06:28:30+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-20T06:28:30+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/ml_concepts.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source-_file': 'ml_concepts.pdf', 'file_type': 'pdf'}, page_content='Machine Learning Concepts\nSupervised Learning: Linear Regression, Logistic Regression\nUnsupervised Learning: K-Means, PCA\nEvaluation Metrics: Accuracy, Precision, Recall'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-20T06:28:30+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-20T06:28:30+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/sample_resume.pdf', 'total_pages': 1, 'page'

In [14]:
#convert text to embeddings
texts = [doc.page_content for doc in chunks]

#generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

#store in the vectodb
vector_store.add_documents(chunks, embeddings)


Generating embeddings for 4 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.29it/s]

Generated embeddings with shape: (4, 384)
Adding 4 documents to vector store...
Successfully added 4 documents to vector store
Total documents in collection: 8


In [15]:
class RAGRetriever:
    """Handles retrieval of RAGs"""

    def __init__(self, vector_store:VectorStore, embedding_manager:EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager


    def retrieve(self, query:str, top_k: int =5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: {query}")
        print(f"Top k: {top_k}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results=top_k,
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata,distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score= 1- distance

                    if(similarity_score > score_threshold):
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents")

            else:
                print(f"No retrieved documents found for query")

            return retrieved_docs
        except Exception as e:
            print(f"Error retrieving documents for query: {e}")
            return []

rag_retriever=RAGRetriever(vector_store,embedding_manager)

In [16]:
rag_retriever

In [17]:
rag_retriever.retrieve("What is machine learning?")

Retrieving documents for query: What is machine learning?
Top k: 5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.99it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents


[{'id': 'doc_1c3e60aa_2',
  'content': 'Artificial Intelligence Basics\nArtificial Intelligence (AI) is the simulation of human intelligence in machines.\nKey areas include Machine Learning, Deep Learning, and NLP.\nApplications include chatbots, recommendation systems, and self-driving cars.',
  'metadata': {'doc_index': 2,
   'creator': '(unspecified)',
   'trapped': '/False',
   'creationdate': '2026-04-20T06:28:30+00:00',
   'page': 0,
   'page_label': '1',
   'source-_file': 'ai_basics.pdf',
   'source': '../data/pdf/ai_basics.pdf',
   'keywords': '',
   'total_pages': 1,
   'content_length': 250,
   'moddate': '2026-04-20T06:28:30+00:00',
   'author': '(anonymous)',
   'title': '(anonymous)',
   'file_type': 'pdf',
   'subject': '(unspecified)',
   'producer': 'ReportLab PDF Library - (opensource)'},
  'similarity_score': 0.06600487232208252,
  'distance': 0.9339951276779175,
  'rank': 1},
 {'id': 'doc_d9a0f4f5_2',
  'content': 'Artificial Intelligence Basics\nArtificial Intellig

In [18]:
import os
from dotenv import load_dotenv
load_dotenv()

print("API key added successfully")


API key added successfully


In [19]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.schema import HumanMessage, SystemMessage


In [20]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM

        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")

        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context

        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length

        Returns:
            Generated response string
        """

        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )

        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)

        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content

        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting

        Args:
            query: User question
            context: Retrieved context

        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""

        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"


In [21]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [22]:
rag_retriever.retrieve("What is machine learning? Explain it properly.")

Retrieving documents for query: What is machine learning? Explain it properly.
Top k: 5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.33it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents


[{'id': 'doc_1c3e60aa_2',
  'content': 'Artificial Intelligence Basics\nArtificial Intelligence (AI) is the simulation of human intelligence in machines.\nKey areas include Machine Learning, Deep Learning, and NLP.\nApplications include chatbots, recommendation systems, and self-driving cars.',
  'metadata': {'content_length': 250,
   'source': '../data/pdf/ai_basics.pdf',
   'producer': 'ReportLab PDF Library - (opensource)',
   'keywords': '',
   'total_pages': 1,
   'title': '(anonymous)',
   'author': '(anonymous)',
   'trapped': '/False',
   'subject': '(unspecified)',
   'creator': '(unspecified)',
   'doc_index': 2,
   'file_type': 'pdf',
   'source-_file': 'ai_basics.pdf',
   'page': 0,
   'creationdate': '2026-04-20T06:28:30+00:00',
   'moddate': '2026-04-20T06:28:30+00:00',
   'page_label': '1'},
  'similarity_score': 0.07321816682815552,
  'distance': 0.9267818331718445,
  'rank': 1},
 {'id': 'doc_d9a0f4f5_2',
  'content': 'Artificial Intelligence Basics\nArtificial Intellig

In [29]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    result=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in result]) if result else ""
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""

    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [36]:
answer = rag_simple("What is machine learning", rag_retriever, llm)
print(answer)

Retrieving documents for query: What is machine learning
Top k: 3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.15it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents
Machine learning is a subset of Artificial Intelligence (AI) that involves training algorithms to learn from data and make predictions or decisions without being explicitly programmed.


In [41]:
def rag_advanced(query:str, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source' : doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

result = rag_advanced("Machine learning models", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: Machine learning models
Top k: 3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.34s/it]


Generated embeddings with shape: (1, 384)
Retrieved 2 documents
Answer: Machine learning models include:

1. Supervised Learning:
   - Linear Regression
   - Logistic Regression

2. Unsupervised Learning:
   - K-Means
   - PCA

3. Evaluation Metrics:
   - Accuracy
   - Precision
   - Recall
Sources: [{'source': '../data/pdf/ml_concepts.pdf', 'page': 0, 'score': 0.129830002784729, 'preview': 'Machine Learning Concepts\nSupervised Learning: Linear Regression, Logistic Regression\nUnsupervised Learning: K-Means, PCA\nEvaluation Metrics: Accuracy, Precision, Recall...'}, {'source': '../data/pdf/ml_concepts.pdf', 'page': 0, 'score': 0.129830002784729, 'preview': 'Machine Learning Concepts\nSupervised Learning: Linear Regression, Logistic Regression\nUnsupervised Learning: K-Means, PCA\nEvaluation Metrics: Accuracy, Precision, Recall...'}]
Confidence: 0.129830002784729
Context Preview: Machine Learning Concepts
Supervised Learning: Linear Regression, Logistic Regression
Unsupervised Learning